## Data Collection and Cleaning for Ulaanbaatar
CS215 Final Project

Arvin Shirchindorj

May 16, 2026

In [ ]:
# Import necessary packages
import os
import time
import requests
import numpy as np
import pandas as pd
from getpass import getpass

## Requesting data

In [ ]:
API_KEY = getpass("Paste your OpenAQ API key here: ")
BASE_URL = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}

Paste your OpenAQ API key here: ··········


In [ ]:
test_url = f"{BASE_URL}/locations"
test_params = {"limit": 5}

response = requests.get(test_url, headers=HEADERS, params=test_params)

if response.status_code == 200:
    print("API connection works.")
else:
    print(response.text)

API connection works.


In [ ]:
# Create folders to keep raw, cleaned, and derived data organized.
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/clean", exist_ok=True)
os.makedirs("data/derived", exist_ok=True)

In [ ]:
# Download location metadata for Mongolia so we can find Ulaanbaatar monitoring sites.
locations_url = f"{BASE_URL}/locations"
params = {
    "limit": 100,
    "country": "MN"
}

response = requests.get(locations_url, headers=HEADERS, params=params)
data = response.json()

rows = []

for loc in data.get("results", []):
    sensors = loc.get("sensors", [])
    sensor_params = []

    # Collect the pollutant names measured at each location.
    for s in sensors:
        param = s.get("parameter")
        if isinstance(param, dict):
            sensor_params.append(param.get("name"))

    coords = loc.get("coordinates", {}) or {}

    rows.append({
        "id": loc.get("id"),
        "name": loc.get("name"),
        "city": loc.get("city"),
        "locality": loc.get("locality"),
        "latitude": coords.get("latitude"),
        "longitude": coords.get("longitude"),
        "parameters": sorted(list(set([p for p in sensor_params if p])))
    })

locations_df = pd.DataFrame(rows)

print("Number of Mongolia locations returned:", len(locations_df))
locations_df

Number of Mongolia locations returned: 100


,id,name,city,locality,latitude,longitude,parameters
0,3,NMA - Nima,None,None,5.583890,-0.199680,"[pm10, pm25]"
1,4,NMT - Nima,None,None,5.581650,-0.198980,"[pm10, pm25]"
2,5,JTA - Jamestown,None,None,5.540114,-0.210397,"[pm10, pm25]"
3,6,ADT - Asylum Down,None,None,5.570722,-0.212056,"[pm10, pm25]"
4,7,ADEPA - Asylum Down,None,None,5.567833,-0.204028,"[pm10, pm25]"
...,...,...,...,...,...,...,...
95,146,Southwark A2 Old Kent Road - UKA00558,None,Southwark,51.480499,-0.059550,"[no2, pm10, pm25]"
96,147,Greenwich - Eltham,None,Greenwich,51.452580,0.070766,"[no2, o3, pm10, pm25]"
97,148,London Bloomsbury - UKA00211,None,Camden,51.522290,-0.125889,"[no2, o3, pm10, pm25, so2]"
98,149,Ealing Horn Lane,None,London,51.518950,-0.265617,"[no2, pm10]"


In [ ]:
# Filter the Mongolia locations down to likely Ulaanbaatar monitoring sites.
# This will help us see which stations belong to the city and what pollutants they measure.

ulaanbaatar_df = locations_df[
    locations_df.astype(str).apply(
        lambda col: col.str.contains("Ulaanbaatar|Ulan Bator|Улаанбаатар", case=False, na=False)
    ).any(axis=1)
].copy()

print("Number of likely Ulaanbaatar locations:", len(ulaanbaatar_df))
ulaanbaatar_df

Number of likely Ulaanbaatar locations: 0


,id,name,city,locality,latitude,longitude,parameters


In [ ]:
# Show all unique city and locality names so we can see how OpenAQ labels Mongolia locations.
# This helps if Ulaanbaatar is spelled differently in the metadata.

print("Unique city values:")
print(sorted(locations_df["city"].dropna().astype(str).unique()))

print("\nUnique locality values:")
print(sorted(locations_df["locality"].dropna().astype(str).unique()))

Unique city values:
[]

Unique locality values:
['Amsterdam', 'Badhoevedorp', 'Beijing', 'Camden', 'Chengdu', 'Chiguayante', 'Concepción', 'Concón', 'Coronel', 'Coyhaique', 'Curicó', 'De Zilk', 'Greenwich', 'Groningen', 'Guangzhou', 'Haarlem', 'Haringey', 'Harrow', 'Heerlen', 'Hellendoorn', 'Hoek v. Holland', 'Houtakker', 'IJmuiden', 'London', 'Machalí', 'Nijmegen', 'Philippine', 'Puente Alto', 'Rotterdam', 'Samut Prakan', 'Santiago', 'Shanghai', 'Shenyang', 'Southwark', 'Talagante', 'Talcahuano', 'Tocopilla', 'Utrecht', 'Valparaíso', 'Wekerom', 'Wieringerwerf', 'Wijk aan Zee', 'Zaandam', 'Zegveld']


This was my initial attempt using country/name filtering. This did not return the expected Ulaanbaatar stations, so I switched to a coordinate-based search.

In [ ]:
# Search for monitoring locations near central Ulaanbaatar using latitude,longitude.

ulaanbaatar_lat = 47.9184
ulaanbaatar_lon = 106.9177

locations_url = f"{BASE_URL}/locations"
params = {
    "coordinates": f"{ulaanbaatar_lat},{ulaanbaatar_lon}",
    "radius": 25000,
    "limit": 100
}

response = requests.get(locations_url, headers=HEADERS, params=params)

print("Status code:", response.status_code)
print("Final URL:", response.url)

if response.status_code != 200:
    print("Response text:", response.text[:500])
else:
    data = response.json()

    rows = []
    for loc in data.get("results", []):
        sensors = loc.get("sensors", [])
        sensor_params = []

        # Collect measured pollutant names for each station.
        for s in sensors:
            param = s.get("parameter")
            if isinstance(param, dict):
                sensor_params.append(param.get("name"))

        coords = loc.get("coordinates", {}) or {}
        country = loc.get("country", {}) or {}
        provider = loc.get("provider", {}) or {}
        owner = loc.get("owner", {}) or {}

        rows.append({
            "location_id": loc.get("id"),
            "name": loc.get("name"),
            "locality": loc.get("locality"),
            "timezone": loc.get("timezone"),
            "country_code": country.get("code"),
            "country_name": country.get("name"),
            "provider": provider.get("name"),
            "owner": owner.get("name"),
            "latitude": coords.get("latitude"),
            "longitude": coords.get("longitude"),
            "datetime_first_utc": (loc.get("datetimeFirst") or {}).get("utc"),
            "datetime_last_utc": (loc.get("datetimeLast") or {}).get("utc"),
            "parameters": sorted(list(set([p for p in sensor_params if p])))
        })

    ub_locations_df = pd.DataFrame(rows)

    print("Number of nearby locations returned:", len(ub_locations_df))
    ub_locations_df.head(20)

Status code: 200
Final URL: https://api.openaq.org/v3/locations?coordinates=47.9184%2C106.9177&radius=25000&limit=100
Number of nearby locations returned: 25


In [ ]:
# Save the Ulaanbaatar-area location metadata so we can reuse it later.
ub_locations_df.to_csv("data/raw/ulaanbaatar_locations.csv", index=False)
print("Saved: data/raw/ulaanbaatar_locations.csv")

Saved: data/raw/ulaanbaatar_locations.csv


In [ ]:
# Show the most important metadata columns so we can inspect the Ulaanbaatar stations.
# This helps us see station names, time coverage, and which pollutants are available.

display_columns = [
    "location_id",
    "name",
    "locality",
    "country_code",
    "latitude",
    "longitude",
    "datetime_first_utc",
    "datetime_last_utc",
    "parameters"
]

ub_locations_df[display_columns].sort_values(by="location_id").reset_index(drop=True)

,location_id,name,locality,country_code,latitude,longitude,datetime_first_utc,datetime_last_utc,parameters
0,19,MNB,None,MN,47.929732,106.888629,2016-01-30T00:30:00Z,2019-03-13T22:30:00Z,"[co, no2, o3, pm10, pm25, so2]"
1,23,Amgalan,None,MN,47.913429,106.997907,2016-01-30T00:30:00Z,2019-03-13T22:30:00Z,"[co, no2, o3, pm10, pm25, so2]"
2,30,100 ail,None,MN,47.932906,106.921383,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z,"[co, no2, o3, pm10, so2]"
3,46,Bukhiin urguu,None,MN,47.917606,106.937361,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z,"[co, no2, o3, pm10, pm25, so2]"
4,48,Baruun 4 zam,None,MN,47.915383,106.894194,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z,"[co, no2, pm10, pm25, so2]"
5,59,Nisekh,None,MN,47.863943,106.779094,2016-02-16T07:00:00Z,2019-03-13T22:30:00Z,"[co, no2, o3, pm10, pm25, so2]"
6,90,Misheel expo,None,MN,47.894339,106.882472,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z,"[co, no2, o3, pm10, so2]"
7,694,Urgakh naran,None,MN,47.866461,107.118019,2016-01-30T00:15:00Z,2019-03-13T21:45:00Z,"[co, no2, o3, pm10, so2]"
8,811,Tolgoit,None,MN,47.922495,106.794805,2016-01-30T00:30:00Z,2019-03-13T22:30:00Z,"[co, no2, o3, pm10, pm25, so2]"
9,2275,Bayankhoshuu,None,MN,47.957560,106.822752,2016-05-25T07:00:00Z,2019-03-13T18:00:00Z,"[pm10, pm25, so2]"


In [ ]:
# Create a simple pollutant coverage table showing how many Ulaanbaatar stations measure each pollutant.
# This will help us decide which pollutants are strong enough to use in the project.

from collections import Counter

parameter_counter = Counter()

for param_list in ub_locations_df["parameters"]:
    if isinstance(param_list, list):
        for p in param_list:
            parameter_counter[p] += 1

parameter_summary_df = pd.DataFrame(
    sorted(parameter_counter.items(), key=lambda x: x[1], reverse=True),
    columns=["parameter", "number_of_stations"]
)

parameter_summary_df

,parameter,number_of_stations
0,no2,17
1,so2,17
2,pm25,15
3,pm10,13
4,co,11
5,o3,8
6,pm1,1
7,relativehumidity,1
8,temperature,1
9,um003,1


In [ ]:
# Create one row per station-parameter pair so we can clearly see
# which Ulaanbaatar stations measure which pollutants.

station_parameter_rows = []

for _, row in ub_locations_df.iterrows():
    location_id = row["location_id"]
    station_name = row["name"]
    locality = row["locality"]
    first_date = row["datetime_first_utc"]
    last_date = row["datetime_last_utc"]
    param_list = row["parameters"]

    if isinstance(param_list, list):
        for param in param_list:
            station_parameter_rows.append({
                "location_id": location_id,
                "station_name": station_name,
                "locality": locality,
                "parameter": param,
                "datetime_first_utc": first_date,
                "datetime_last_utc": last_date
            })

station_parameter_df = pd.DataFrame(station_parameter_rows)

print("Number of station-parameter rows:", len(station_parameter_df))
station_parameter_df.sort_values(["parameter", "location_id"]).reset_index(drop=True)

Number of station-parameter rows: 85


,location_id,station_name,locality,parameter,datetime_first_utc,datetime_last_utc
0,19,MNB,None,co,2016-01-30T00:30:00Z,2019-03-13T22:30:00Z
1,23,Amgalan,None,co,2016-01-30T00:30:00Z,2019-03-13T22:30:00Z
2,30,100 ail,None,co,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z
3,46,Bukhiin urguu,None,co,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z
4,48,Baruun 4 zam,None,co,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z
...,...,...,...,...,...,...
80,7126,УБ-9,Ulaanbaatar,so2,2016-12-26T00:00:00Z,2019-11-30T00:00:00Z
81,7215,УБ-6,Ulaanbaatar,so2,2017-05-25T00:00:00Z,2019-12-31T00:00:00Z
82,7230,Зуунмод,Зуунмод хот,so2,2017-10-16T00:00:00Z,2019-12-31T12:00:00Z
83,6119008,Bumbugur,None,temperature,2025-11-04T11:00:00Z,2026-05-19T02:00:00Z


In [ ]:
# Keep only the main air pollutants

main_parameters = ["pm25", "pm10", "no2", "so2", "co", "o3"]

main_station_parameter_df = station_parameter_df[
    station_parameter_df["parameter"].isin(main_parameters)
].copy()

main_station_parameter_df.sort_values(["parameter", "location_id"]).reset_index(drop=True)

,location_id,station_name,locality,parameter,datetime_first_utc,datetime_last_utc
0,19,MNB,None,co,2016-01-30T00:30:00Z,2019-03-13T22:30:00Z
1,23,Amgalan,None,co,2016-01-30T00:30:00Z,2019-03-13T22:30:00Z
2,30,100 ail,None,co,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z
3,46,Bukhiin urguu,None,co,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z
4,48,Baruun 4 zam,None,co,2016-01-30T01:00:00Z,2019-03-13T22:00:00Z
...,...,...,...,...,...,...
76,6523,1-r khoroolol,None,so2,2019-03-01T07:45:00Z,2019-03-13T21:45:00Z
77,7035,УБ-11,Ulaanbaatar,so2,2016-12-24T00:00:00Z,2019-12-31T00:00:00Z
78,7126,УБ-9,Ulaanbaatar,so2,2016-12-26T00:00:00Z,2019-11-30T00:00:00Z
79,7215,УБ-6,Ulaanbaatar,so2,2017-05-25T00:00:00Z,2019-12-31T00:00:00Z


In [ ]:
# Save the station-parameter table for later reference.

main_station_parameter_df.to_csv("data/raw/ulaanbaatar_station_parameter_table.csv", index=False)

## Build sensor-level metadata

In the previous step, I found the monitoring locations near Ulaanbaatar and looked at which pollutants were available at each station. That gave me a good overview of the data, but it was still only at the location level.

For the actual data collection step, I need more detailed metadata at the sensor level. This is necessary because one location can have multiple sensors, and each sensor may measure a different pollutant such as PM2.5, PM10, NO2, or SO2. To download the measurement data correctly, I need the specific sensor IDs associated with each pollutant.

In [ ]:
# Re-run the Ulaanbaatar location search and keep full sensor-level metadata.

ulaanbaatar_lat = 47.9184
ulaanbaatar_lon = 106.9177

locations_url = f"{BASE_URL}/locations"
params = {
    "coordinates": f"{ulaanbaatar_lat},{ulaanbaatar_lon}",
    "radius": 25000,
    "limit": 100
}

response = requests.get(locations_url, headers=HEADERS, params=params)

print("Status code:", response.status_code)

data = response.json()

sensor_rows = []

for loc in data.get("results", []):
    location_id = loc.get("id")
    station_name = loc.get("name")
    locality = loc.get("locality")
    timezone = loc.get("timezone")

    coords = loc.get("coordinates", {}) or {}
    latitude = coords.get("latitude")
    longitude = coords.get("longitude")

    sensors = loc.get("sensors", [])

    # Keep one row per sensor so we can download measurements later.
    for sensor in sensors:
        parameter_info = sensor.get("parameter", {}) or {}

        sensor_rows.append({
            "location_id": location_id,
            "station_name": station_name,
            "locality": locality,
            "timezone": timezone,
            "latitude": latitude,
            "longitude": longitude,
            "sensor_id": sensor.get("id"),
            "parameter": parameter_info.get("name"),
            "units": sensor.get("units"),
            "datetime_first_utc": (sensor.get("datetimeFirst") or {}).get("utc"),
            "datetime_last_utc": (sensor.get("datetimeLast") or {}).get("utc")
        })

sensor_metadata_df = pd.DataFrame(sensor_rows)

print("Number of sensor rows:", len(sensor_metadata_df))
sensor_metadata_df.head(20)

Status code: 200
Number of sensor rows: 85


,location_id,station_name,locality,timezone,latitude,longitude,sensor_id,parameter,units,datetime_first_utc,datetime_last_utc
0,19,MNB,None,Asia/Ulaanbaatar,47.929732,106.888629,5136,co,None,None,None
1,19,MNB,None,Asia/Ulaanbaatar,47.929732,106.888629,5137,no2,None,None,None
2,19,MNB,None,Asia/Ulaanbaatar,47.929732,106.888629,38,o3,None,None,None
3,19,MNB,None,Asia/Ulaanbaatar,47.929732,106.888629,5138,pm10,None,None,None
4,19,MNB,None,Asia/Ulaanbaatar,47.929732,106.888629,5139,pm25,None,None,None
5,19,MNB,None,Asia/Ulaanbaatar,47.929732,106.888629,5140,so2,None,None,None
6,23,Amgalan,None,Asia/Ulaanbaatar,47.913429,106.997907,68,co,None,None,None
7,23,Amgalan,None,Asia/Ulaanbaatar,47.913429,106.997907,5177,no2,None,None,None
8,23,Amgalan,None,Asia/Ulaanbaatar,47.913429,106.997907,5178,o3,None,None,None
9,23,Amgalan,None,Asia/Ulaanbaatar,47.913429,106.997907,5183,pm10,None,None,None


In [ ]:
# Keep only the main pollutants that are strong enough for the project.

main_parameters = ["pm25", "pm10", "no2", "so2", "co", "o3"]

main_sensor_metadata_df = sensor_metadata_df[
    sensor_metadata_df["parameter"].isin(main_parameters)
].copy()

print("Number of main-pollutant sensor rows:", len(main_sensor_metadata_df))
main_sensor_metadata_df.sort_values(["parameter", "location_id", "sensor_id"]).reset_index(drop=True)

Number of main-pollutant sensor rows: 81


,location_id,station_name,locality,timezone,latitude,longitude,sensor_id,parameter,units,datetime_first_utc,datetime_last_utc
0,19,MNB,None,Asia/Ulaanbaatar,47.929732,106.888629,5136,co,None,None,None
1,23,Amgalan,None,Asia/Ulaanbaatar,47.913429,106.997907,68,co,None,None,None
2,30,100 ail,None,Asia/Ulaanbaatar,47.932906,106.921383,5373,co,None,None,None
3,46,Bukhiin urguu,None,Asia/Ulaanbaatar,47.917606,106.937361,1454,co,None,None,None
4,48,Baruun 4 zam,None,Asia/Ulaanbaatar,47.915383,106.894194,108,co,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
76,6523,1-r khoroolol,None,Asia/Ulaanbaatar,47.917981,106.848061,18416,so2,None,None,None
77,7035,УБ-11,Ulaanbaatar,Asia/Ulaanbaatar,47.951431,106.904072,20773,so2,None,None,None
78,7126,УБ-9,Ulaanbaatar,Asia/Ulaanbaatar,47.981914,106.940511,20552,so2,None,None,None
79,7215,УБ-6,Ulaanbaatar,Asia/Ulaanbaatar,47.913450,106.972031,20776,so2,None,None,None


In [ ]:
# Save the sensor-level metadata table for later use.

main_sensor_metadata_df.to_csv("data/raw/ulaanbaatar_sensor_metadata.csv", index=False)

Now that I have a sensor-level metadata table, the next step is to test the actual data download. I am starting with a small sample first instead of downloading everything at once. It will allow me to make sure the API endpoint works, the columns look correct, and the data is in a format I can actually use later.

In [ ]:
# Select a small set of sensors to test the daily measurements endpoint.

test_sensor_ids = main_sensor_metadata_df["sensor_id"].dropna().astype(int).unique()[:3]
print("Test sensor IDs:", test_sensor_ids)

Test sensor IDs: [5136 5137   38]


In [ ]:
# Download daily measurement data for a few test sensors.
# For now, I am using a limited date range so the response stays small and easy to inspect.

test_daily_rows = []

for sensor_id in test_sensor_ids:
    daily_url = f"{BASE_URL}/sensors/{sensor_id}/days"
    params = {
        "date_from": "2018-01-01",
        "date_to": "2018-12-31",
        "limit": 1000
    }

    response = requests.get(daily_url, headers=HEADERS, params=params)

    print(f"Sensor {sensor_id} status code:", response.status_code)

    if response.status_code != 200:
        print(response.text[:300])
        continue

    data = response.json()

    for row in data.get("results", []):
        test_daily_rows.append({
            "sensor_id": sensor_id,
            "datetime_utc": (row.get("period") or {}).get("datetimeFrom", {}).get("utc"),
            "value": row.get("value"),
            "unit": row.get("unit"),
            "summary": row.get("summary"),
            "coverage": row.get("coverage"),
            "parameter": row.get("parameter", {}).get("name") if isinstance(row.get("parameter"), dict) else None
        })

test_daily_df = pd.DataFrame(test_daily_rows)

print("Number of test daily rows:", len(test_daily_df))
test_daily_df.head(20)

Sensor 5136 status code: 200
Sensor 5137 status code: 200
Sensor 38 status code: 200
Number of test daily rows: 681


,sensor_id,datetime_utc,value,unit,summary,coverage,parameter
0,5136,2017-12-31T16:00:00Z,4260.0,None,"{'min': 1453.0, 'q02': 1485.0, 'q25': 2844.0, ...","{'expectedCount': 24, 'expectedInterval': '24:...",co
1,5136,2018-01-01T16:00:00Z,1910.0,None,"{'min': 700.0, 'q02': 703.52, 'q25': 1262.25, ...","{'expectedCount': 24, 'expectedInterval': '24:...",co
2,5136,2018-01-02T16:00:00Z,4300.0,None,"{'min': 746.0, 'q02': 777.6, 'q25': 1858.0, 'm...","{'expectedCount': 24, 'expectedInterval': '24:...",co
3,5136,2018-01-03T16:00:00Z,5410.0,None,"{'min': 1309.0, 'q02': 1572.97, 'q25': 2557.75...","{'expectedCount': 24, 'expectedInterval': '24:...",co
4,5136,2018-01-04T16:00:00Z,6210.0,None,"{'min': 1638.0, 'q02': 1651.5, 'q25': 1968.75,...","{'expectedCount': 24, 'expectedInterval': '24:...",co
5,5136,2018-01-05T16:00:00Z,4060.0,None,"{'min': 1377.0, 'q02': 1511.96, 'q25': 2563.0,...","{'expectedCount': 24, 'expectedInterval': '24:...",co
6,5136,2018-01-06T16:00:00Z,1640.0,None,"{'min': 657.0, 'q02': 674.64, 'q25': 1048.5, '...","{'expectedCount': 24, 'expectedInterval': '24:...",co
7,5136,2018-01-07T16:00:00Z,1200.0,None,"{'min': 362.0, 'q02': 383.6, 'q25': 643.0, 'me...","{'expectedCount': 24, 'expectedInterval': '24:...",co
8,5136,2018-01-08T16:00:00Z,1390.0,None,"{'min': 961.0, 'q02': 965.88, 'q25': 1022.0, '...","{'expectedCount': 24, 'expectedInterval': '24:...",co
9,5136,2018-01-09T16:00:00Z,3740.0,None,"{'min': 706.0, 'q02': 736.4, 'q25': 1028.0, 'm...","{'expectedCount': 24, 'expectedInterval': '24:...",co


In [ ]:
# Attach station and pollutant metadata to the downloaded test rows.
# This makes the test data easier to read and helps confirm that the right sensors were pulled.

test_daily_df = test_daily_df.merge(
    main_sensor_metadata_df[["sensor_id", "location_id", "station_name", "parameter"]],
    on="sensor_id",
    how="left",
    suffixes=("", "_metadata")
)

test_daily_df.head(20)

,sensor_id,datetime_utc,value,unit,summary,coverage,parameter,location_id,station_name,parameter_metadata
0,5136,2017-12-31T16:00:00Z,4260.0,None,"{'min': 1453.0, 'q02': 1485.0, 'q25': 2844.0, ...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
1,5136,2018-01-01T16:00:00Z,1910.0,None,"{'min': 700.0, 'q02': 703.52, 'q25': 1262.25, ...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
2,5136,2018-01-02T16:00:00Z,4300.0,None,"{'min': 746.0, 'q02': 777.6, 'q25': 1858.0, 'm...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
3,5136,2018-01-03T16:00:00Z,5410.0,None,"{'min': 1309.0, 'q02': 1572.97, 'q25': 2557.75...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
4,5136,2018-01-04T16:00:00Z,6210.0,None,"{'min': 1638.0, 'q02': 1651.5, 'q25': 1968.75,...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
5,5136,2018-01-05T16:00:00Z,4060.0,None,"{'min': 1377.0, 'q02': 1511.96, 'q25': 2563.0,...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
6,5136,2018-01-06T16:00:00Z,1640.0,None,"{'min': 657.0, 'q02': 674.64, 'q25': 1048.5, '...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
7,5136,2018-01-07T16:00:00Z,1200.0,None,"{'min': 362.0, 'q02': 383.6, 'q25': 643.0, 'me...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
8,5136,2018-01-08T16:00:00Z,1390.0,None,"{'min': 961.0, 'q02': 965.88, 'q25': 1022.0, '...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co
9,5136,2018-01-09T16:00:00Z,3740.0,None,"{'min': 706.0, 'q02': 736.4, 'q25': 1028.0, 'm...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB,co


In [ ]:
# Save the small test dataset so we can refer back to it later if needed.

test_daily_df.to_csv("data/raw/test_daily_measurements.csv", index=False)

The test download worked, which means I can now pull daily air quality data from the selected OpenAQ sensors.

One issue I noticed is that the unit field in the downloaded daily data is missing in this sample. Because of that, I will use the unit information from the sensor metadata table when I build the full dataset. This will help keep the final air quality data more complete and easier to interpret later.

In [ ]:
# Fill in missing units using the sensor metadata table.
# This is useful because the daily endpoint did not return units in the sample download.

test_daily_df = test_daily_df.drop(columns=["parameter_metadata"], errors="ignore")

test_daily_df = test_daily_df.merge(
    main_sensor_metadata_df[["sensor_id", "units"]],
    on="sensor_id",
    how="left",
    suffixes=("", "_from_metadata")
)

# If the downloaded unit is missing, use the metadata unit instead.
test_daily_df["unit"] = test_daily_df["unit"].fillna(test_daily_df["units"])
test_daily_df = test_daily_df.drop(columns=["units"])

test_daily_df.head(10)

,sensor_id,datetime_utc,value,unit,summary,coverage,parameter,location_id,station_name
0,5136,2017-12-31T16:00:00Z,4260.0,None,"{'min': 1453.0, 'q02': 1485.0, 'q25': 2844.0, ...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
1,5136,2018-01-01T16:00:00Z,1910.0,None,"{'min': 700.0, 'q02': 703.52, 'q25': 1262.25, ...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
2,5136,2018-01-02T16:00:00Z,4300.0,None,"{'min': 746.0, 'q02': 777.6, 'q25': 1858.0, 'm...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
3,5136,2018-01-03T16:00:00Z,5410.0,None,"{'min': 1309.0, 'q02': 1572.97, 'q25': 2557.75...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
4,5136,2018-01-04T16:00:00Z,6210.0,None,"{'min': 1638.0, 'q02': 1651.5, 'q25': 1968.75,...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
5,5136,2018-01-05T16:00:00Z,4060.0,None,"{'min': 1377.0, 'q02': 1511.96, 'q25': 2563.0,...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
6,5136,2018-01-06T16:00:00Z,1640.0,None,"{'min': 657.0, 'q02': 674.64, 'q25': 1048.5, '...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
7,5136,2018-01-07T16:00:00Z,1200.0,None,"{'min': 362.0, 'q02': 383.6, 'q25': 643.0, 'me...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
8,5136,2018-01-08T16:00:00Z,1390.0,None,"{'min': 961.0, 'q02': 965.88, 'q25': 1022.0, '...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB
9,5136,2018-01-09T16:00:00Z,3740.0,None,"{'min': 706.0, 'q02': 736.4, 'q25': 1028.0, 'm...","{'expectedCount': 24, 'expectedInterval': '24:...",co,19,MNB


In the test download, the measurement values were returned correctly, but the unit field was missing. I checked the sensor-level metadata for the same test sensors, and the unit values were also missing there. This suggests that the missing units are coming from the source metadata rather than from my code.

Because of that, I will continue with the full download and keep track of this as a data limitation. If needed, I can later infer or document likely units for the main pollutants.

In [ ]:
# Select the main pollutant sensors that will be used in the full Ulaanbaatar download.

main_parameters = ["pm25", "pm10", "no2", "so2", "co", "o3"]

full_sensor_df = main_sensor_metadata_df[
    main_sensor_metadata_df["parameter"].isin(main_parameters)
].copy()

print("Number of sensors selected for full download:", full_sensor_df["sensor_id"].nunique())
full_sensor_df.sort_values(["parameter", "location_id", "sensor_id"]).reset_index(drop=True).head(20)

Number of sensors selected for full download: 81


,location_id,station_name,locality,timezone,latitude,longitude,sensor_id,parameter,units,datetime_first_utc,datetime_last_utc
0,19,MNB,None,Asia/Ulaanbaatar,47.929732,106.888629,5136,co,None,None,None
1,23,Amgalan,None,Asia/Ulaanbaatar,47.913429,106.997907,68,co,None,None,None
2,30,100 ail,None,Asia/Ulaanbaatar,47.932906,106.921383,5373,co,None,None,None
3,46,Bukhiin urguu,None,Asia/Ulaanbaatar,47.917606,106.937361,1454,co,None,None,None
4,48,Baruun 4 zam,None,Asia/Ulaanbaatar,47.915383,106.894194,108,co,None,None,None
5,59,Nisekh,None,Asia/Ulaanbaatar,47.863943,106.779094,1513,co,None,None,None
6,90,Misheel expo,None,Asia/Ulaanbaatar,47.894339,106.882472,13218,co,None,None,None
7,694,Urgakh naran,None,Asia/Ulaanbaatar,47.866461,107.118019,1195,co,None,None,None
8,811,Tolgoit,None,Asia/Ulaanbaatar,47.922495,106.794805,5122,co,None,None,None
9,2590,Mongol gazar,None,Asia/Ulaanbaatar,47.903539,106.850708,5371,co,None,None,None


In [ ]:
# Download daily air quality data for all selected Ulaanbaatar sensors.

full_daily_rows = []

total_sensors = len(full_sensor_df)

for i, (_, sensor_row) in enumerate(full_sensor_df.iterrows(), start=1):
    sensor_id = int(sensor_row["sensor_id"])
    station_name = sensor_row["station_name"]
    location_id = sensor_row["location_id"]
    parameter = sensor_row["parameter"]
    unit = sensor_row["units"]

    daily_url = f"{BASE_URL}/sensors/{sensor_id}/days"
    params = {
        "date_from": "2016-01-01",
        "date_to": "2019-12-31",
        "limit": 2000
    }

    response = requests.get(daily_url, headers=HEADERS, params=params)

    print(f"[{i}/{total_sensors}] Sensor {sensor_id} ({parameter}, {station_name}) status code: {response.status_code}")

    if response.status_code != 200:
        print("Skipped because request failed.")
        continue

    data = response.json()

    for row in data.get("results", []):
        full_daily_rows.append({
            "sensor_id": sensor_id,
            "location_id": location_id,
            "station_name": station_name,
            "parameter": parameter,
            "unit": unit,
            "datetime_utc": (row.get("period") or {}).get("datetimeFrom", {}).get("utc"),
            "value": row.get("value"),
            "summary": row.get("summary"),
            "coverage": row.get("coverage")
        })

full_daily_df = pd.DataFrame(full_daily_rows)

print("Number of full daily rows:", len(full_daily_df))
full_daily_df.head(10)

[1/81] Sensor 5136 (co, MNB) status code: 422
Skipped because request failed.
[2/81] Sensor 5137 (no2, MNB) status code: 422
Skipped because request failed.
[3/81] Sensor 38 (o3, MNB) status code: 422
Skipped because request failed.
[4/81] Sensor 5138 (pm10, MNB) status code: 422
Skipped because request failed.
[5/81] Sensor 5139 (pm25, MNB) status code: 422
Skipped because request failed.
[6/81] Sensor 5140 (so2, MNB) status code: 422
Skipped because request failed.
[7/81] Sensor 68 (co, Amgalan) status code: 422
Skipped because request failed.
[8/81] Sensor 5177 (no2, Amgalan) status code: 422
Skipped because request failed.
[9/81] Sensor 5178 (o3, Amgalan) status code: 422
Skipped because request failed.
[10/81] Sensor 5183 (pm10, Amgalan) status code: 422
Skipped because request failed.
[11/81] Sensor 42 (pm25, Amgalan) status code: 422
Skipped because request failed.
[12/81] Sensor 5186 (so2, Amgalan) status code: 422
Skipped because request failed.
[13/81] Sensor 5373 (co, 100 ai

""


In [ ]:
# Test one failed sensor directly and print the full validation error message.

failed_sensor_id = 18416  # replace with any failed sensor you saw
failed_url = f"{BASE_URL}/sensors/{failed_sensor_id}/days"
failed_params = {
    "date_from": "2016-01-01",
    "date_to": "2019-12-31",
    "limit": 1000
}

failed_response = requests.get(failed_url, headers=HEADERS, params=failed_params)

print("Status code:", failed_response.status_code)
print("Final URL:", failed_response.url)
print("Response text:")
print(failed_response.text)

Status code: 429
Final URL: https://api.openaq.org/v3/sensors/18416/days?date_from=2016-01-01&date_to=2019-12-31&limit=1000
Response text:
{"detail":"Too many requests"}


The failed-sensor test showed that at least one sensor that returned an error in the large loop actually works when requested by itself. I figured the problem is not the sensor itself, but the way the full loop is sending many requests in a row.

To make the download process more stable, I will retry the full pull with a short pause between requests and keep track of any sensors that still fail. This will make the data collection step more reliable and easier to debug.

In [ ]:
# Download daily air quality data for all selected Ulaanbaatar sensors
# with a short pause and retry logic to reduce API failures.

import time

full_daily_rows = []
failed_requests = []

total_sensors = len(full_sensor_df)

for i, (_, sensor_row) in enumerate(full_sensor_df.iterrows(), start=1):
    sensor_id = int(sensor_row["sensor_id"])
    station_name = sensor_row["station_name"]
    location_id = sensor_row["location_id"]
    parameter = sensor_row["parameter"]
    unit = sensor_row["units"]

    daily_url = f"{BASE_URL}/sensors/{sensor_id}/days"
    params = {
        "date_from": "2016-01-01",
        "date_to": "2019-12-31",
        "limit": 1000
    }

    success = False

    for attempt in range(1, 4):
        response = requests.get(daily_url, headers=HEADERS, params=params)

        print(f"[{i}/{total_sensors}] Sensor {sensor_id} ({parameter}, {station_name}) | attempt {attempt} | status {response.status_code}")

        if response.status_code == 200:
            data = response.json()

            for row in data.get("results", []):
                full_daily_rows.append({
                    "sensor_id": sensor_id,
                    "location_id": location_id,
                    "station_name": station_name,
                    "parameter": parameter,
                    "unit": unit,
                    "datetime_utc": (row.get("period") or {}).get("datetimeFrom", {}).get("utc"),
                    "value": row.get("value"),
                    "summary": row.get("summary"),
                    "coverage": row.get("coverage")
                })

            success = True
            break

        time.sleep(2)

    if not success:
        failed_requests.append({
            "sensor_id": sensor_id,
            "location_id": location_id,
            "station_name": station_name,
            "parameter": parameter,
            "status_code": response.status_code,
            "response_text": response.text[:300]
        })

    time.sleep(1)

full_daily_df = pd.DataFrame(full_daily_rows)
failed_requests_df = pd.DataFrame(failed_requests)

print("Number of full daily rows:", len(full_daily_df))
print("Number of failed sensor requests:", len(failed_requests_df))

[1/81] Sensor 5136 (co, MNB) | attempt 1 | status 429
[1/81] Sensor 5136 (co, MNB) | attempt 2 | status 429
[1/81] Sensor 5136 (co, MNB) | attempt 3 | status 429
[2/81] Sensor 5137 (no2, MNB) | attempt 1 | status 429
[2/81] Sensor 5137 (no2, MNB) | attempt 2 | status 429
[2/81] Sensor 5137 (no2, MNB) | attempt 3 | status 429
[3/81] Sensor 38 (o3, MNB) | attempt 1 | status 429
[3/81] Sensor 38 (o3, MNB) | attempt 2 | status 200
[4/81] Sensor 5138 (pm10, MNB) | attempt 1 | status 200
[5/81] Sensor 5139 (pm25, MNB) | attempt 1 | status 200
[6/81] Sensor 5140 (so2, MNB) | attempt 1 | status 200
[7/81] Sensor 68 (co, Amgalan) | attempt 1 | status 200
[8/81] Sensor 5177 (no2, Amgalan) | attempt 1 | status 200
[9/81] Sensor 5178 (o3, Amgalan) | attempt 1 | status 200
[10/81] Sensor 5183 (pm10, Amgalan) | attempt 1 | status 200
[11/81] Sensor 42 (pm25, Amgalan) | attempt 1 | status 200
[12/81] Sensor 5186 (so2, Amgalan) | attempt 1 | status 200
[13/81] Sensor 5373 (co, 100 ail) | attempt 1 | s

In [ ]:
# Save the successful download rows and the failed request log.

full_daily_df.to_csv("data/raw/ulaanbaatar_air_daily_raw.csv", index=False)

## Inspect and clean the raw air quality data

Now that I have successfully downloaded the full daily air quality dataset for Ulaanbaatar, the next step is to inspect and clean it.

In [ ]:
# Inspect the basic structure of the raw air quality dataset.

print("Shape of raw air dataset:", full_daily_df.shape)
print("\nColumns:")
print(full_daily_df.columns.tolist())

print("\nFirst 5 rows:")
display(full_daily_df.head())

print("\nData types:")
print(full_daily_df.dtypes)

Shape of raw air dataset: (50642, 9)

Columns:
['sensor_id', 'location_id', 'station_name', 'parameter', 'unit', 'datetime_utc', 'value', 'summary', 'coverage']

First 5 rows:


,sensor_id,location_id,station_name,parameter,unit,datetime_utc,value,summary,coverage
0,38,19,MNB,o3,None,2016-01-29T16:00:00Z,25.30,"{'min': 10.5, 'q02': 11.620000000000001, 'q25'...","{'expectedCount': 24, 'expectedInterval': '24:..."
1,38,19,MNB,o3,None,2016-01-31T16:00:00Z,9.43,"{'min': 4.0, 'q02': 4.42, 'q25': 6.0, 'median'...","{'expectedCount': 24, 'expectedInterval': '24:..."
2,38,19,MNB,o3,None,2016-02-01T16:00:00Z,39.50,"{'min': 4.5, 'q02': 4.5, 'q25': 5.0, 'median':...","{'expectedCount': 24, 'expectedInterval': '24:..."
3,38,19,MNB,o3,None,2016-02-02T16:00:00Z,154.00,"{'min': 45.5, 'q02': 48.26, 'q25': 86.125, 'me...","{'expectedCount': 24, 'expectedInterval': '24:..."
4,38,19,MNB,o3,None,2016-02-03T16:00:00Z,136.00,"{'min': 55.5, 'q02': 57.34, 'q25': 74.125, 'me...","{'expectedCount': 24, 'expectedInterval': '24:..."



Data types:
sensor_id         int64
location_id       int64
station_name     object
parameter        object
unit             object
datetime_utc     object
value           float64
summary          object
coverage         object
dtype: object


In [ ]:
# Check missing values and duplicates using only hashable columns.
# The full dataframe includes dictionary columns, so we avoid using those for duplicate checks.

missing_summary = full_daily_df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
print(missing_summary)

duplicate_subset = ["sensor_id", "location_id", "station_name", "parameter", "datetime_utc", "value"]
duplicate_count = full_daily_df.duplicated(subset=duplicate_subset).sum()

print("\nNumber of duplicate rows based on key columns:", duplicate_count)

Missing values by column:
unit            50642
value             157
sensor_id           0
station_name        0
location_id         0
parameter           0
datetime_utc        0
summary             0
coverage            0
dtype: int64

Number of duplicate rows based on key columns: 0


In [ ]:
# Create a cleaned copy of the air quality data and convert datetime fields.

air_clean_df = full_daily_df.copy()

air_clean_df["datetime_utc"] = pd.to_datetime(air_clean_df["datetime_utc"], errors="coerce")
air_clean_df["date"] = air_clean_df["datetime_utc"].dt.date
air_clean_df["year"] = air_clean_df["datetime_utc"].dt.year
air_clean_df["month"] = air_clean_df["datetime_utc"].dt.month
air_clean_df["day"] = air_clean_df["datetime_utc"].dt.day

# Create a season variable for later seasonal analysis.
def month_to_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    elif month in [9, 10, 11]:
        return "Fall"
    return None

air_clean_df["season"] = air_clean_df["month"].apply(month_to_season)

print("Datetime conversion complete.")
display(air_clean_df.head())

Datetime conversion complete.


,sensor_id,location_id,station_name,parameter,unit,datetime_utc,value,summary,coverage,date,year,month,day,season
0,38,19,MNB,o3,None,2016-01-29 16:00:00+00:00,25.30,"{'min': 10.5, 'q02': 11.620000000000001, 'q25'...","{'expectedCount': 24, 'expectedInterval': '24:...",2016-01-29,2016,1,29,Winter
1,38,19,MNB,o3,None,2016-01-31 16:00:00+00:00,9.43,"{'min': 4.0, 'q02': 4.42, 'q25': 6.0, 'median'...","{'expectedCount': 24, 'expectedInterval': '24:...",2016-01-31,2016,1,31,Winter
2,38,19,MNB,o3,None,2016-02-01 16:00:00+00:00,39.50,"{'min': 4.5, 'q02': 4.5, 'q25': 5.0, 'median':...","{'expectedCount': 24, 'expectedInterval': '24:...",2016-02-01,2016,2,1,Winter
3,38,19,MNB,o3,None,2016-02-02 16:00:00+00:00,154.00,"{'min': 45.5, 'q02': 48.26, 'q25': 86.125, 'me...","{'expectedCount': 24, 'expectedInterval': '24:...",2016-02-02,2016,2,2,Winter
4,38,19,MNB,o3,None,2016-02-03 16:00:00+00:00,136.00,"{'min': 55.5, 'q02': 57.34, 'q25': 74.125, 'me...","{'expectedCount': 24, 'expectedInterval': '24:...",2016-02-03,2016,2,3,Winter


In [ ]:
# Check whether any datetime values failed to convert and inspect the date range.

invalid_datetime_rows = air_clean_df["datetime_utc"].isna().sum()
print("Rows with invalid datetime values:", invalid_datetime_rows)

print("\nDate range:")
print("Start:", air_clean_df["datetime_utc"].min())
print("End:", air_clean_df["datetime_utc"].max())

Rows with invalid datetime values: 0

Date range:
Start: 2016-01-29 16:00:00+00:00
End: 2019-12-30 16:00:00+00:00


In [ ]:
# Drop duplicates using the main identifying columns only.

air_clean_df = air_clean_df.drop_duplicates(subset=duplicate_subset).copy()

print("Shape after dropping duplicates:", air_clean_df.shape)

Shape after dropping duplicates: (50642, 14)


In [ ]:
# Remove rows where the pollutant value is missing, since those cannot be used in analysis.

air_clean_df = air_clean_df.dropna(subset=["value"]).copy()

print("Shape after dropping rows with missing values:", air_clean_df.shape)

Shape after dropping rows with missing values: (50485, 14)


In [ ]:
# Save the cleaned air quality dataset.

air_clean_df.to_csv("data/clean/ulaanbaatar_air_daily_clean.csv", index=False)

The dataset includes two nested columns called "summary" and "coverage". These contain extra information about each daily measurement, such as minimum and maximum values, percentile-style summary values, and how complete the daily record is.
Now I am unpacking these dictionary columns into separate variables. This will make the dataset easier to analyze later and will also let me use coverage information to judge how reliable each daily observation is.

In [ ]:
# Expand the nested summary and coverage dictionaries into separate columns.


summary_df = pd.json_normalize(air_clean_df["summary"])
coverage_df = pd.json_normalize(air_clean_df["coverage"])

# Rename the expanded columns so their meaning stays clear.
summary_df.columns = [f"summary_{col}" for col in summary_df.columns]
coverage_df.columns = [f"coverage_{col}" for col in coverage_df.columns]

# Combine the expanded columns back into the cleaned air dataframe.
air_clean_df = pd.concat(
    [air_clean_df.reset_index(drop=True), summary_df, coverage_df],
    axis=1
)

print("Expanded summary and coverage columns added.")
print("\nNew columns:")
print(air_clean_df.columns.tolist())

Expanded summary and coverage columns added.

New columns:
['sensor_id', 'location_id', 'station_name', 'parameter', 'unit', 'datetime_utc', 'value', 'summary', 'coverage', 'date', 'year', 'month', 'day', 'season', 'summary_min', 'summary_q02', 'summary_q25', 'summary_median', 'summary_q75', 'summary_q98', 'summary_max', 'summary_avg', 'summary_sd', 'coverage_expectedCount', 'coverage_expectedInterval', 'coverage_observedCount', 'coverage_observedInterval', 'coverage_percentComplete', 'coverage_percentCoverage', 'coverage_datetimeFrom.utc', 'coverage_datetimeFrom.local', 'coverage_datetimeTo.utc', 'coverage_datetimeTo.local']


In [ ]:
# Preview the main expanded summary and coverage fields.

expanded_cols = [
    col for col in air_clean_df.columns
    if col.startswith("summary_") or col.startswith("coverage_")
]

print("Expanded metadata columns:")
print(expanded_cols)

display(air_clean_df[expanded_cols].head())

Expanded metadata columns:
['summary_min', 'summary_q02', 'summary_q25', 'summary_median', 'summary_q75', 'summary_q98', 'summary_max', 'summary_avg', 'summary_sd', 'coverage_expectedCount', 'coverage_expectedInterval', 'coverage_observedCount', 'coverage_observedInterval', 'coverage_percentComplete', 'coverage_percentCoverage', 'coverage_datetimeFrom.utc', 'coverage_datetimeFrom.local', 'coverage_datetimeTo.utc', 'coverage_datetimeTo.local']


,summary_min,summary_q02,summary_q25,summary_median,summary_q75,summary_q98,summary_max,summary_avg,summary_sd,coverage_expectedCount,coverage_expectedInterval,coverage_observedCount,coverage_observedInterval,coverage_percentComplete,coverage_percentCoverage,coverage_datetimeFrom.utc,coverage_datetimeFrom.local,coverage_datetimeTo.utc,coverage_datetimeTo.local
0,10.5,11.62,20.750,26.75,31.625,34.58,35.0,25.312500,8.250271,24,24:00:00,8,08:00:00,33.0,33.0,2016-01-30T01:00:00Z,2016-01-30T09:00:00+08:00,2016-01-30T08:00:00Z,2016-01-30T16:00:00+08:00
1,4.0,4.42,6.000,8.00,13.000,17.22,17.5,9.433333,4.396969,24,24:00:00,15,15:00:00,63.0,63.0,2016-02-01T02:00:00Z,2016-02-01T10:00:00+08:00,2016-02-01T16:00:00Z,2016-02-02T00:00:00+08:00
2,4.5,4.50,5.000,7.00,67.250,202.89,264.0,39.454545,61.395100,24,24:00:00,24,24:00:00,100.0,100.0,2016-02-01T16:30:00Z,2016-02-02T00:30:00+08:00,2016-02-02T16:00:00Z,2016-02-03T00:00:00+08:00
3,45.5,48.26,86.125,111.25,188.375,390.53,411.0,154.208333,157.093147,24,24:00:00,24,24:00:00,100.0,100.0,2016-02-02T16:30:00Z,2016-02-03T00:30:00+08:00,2016-02-03T16:00:00Z,2016-02-04T00:00:00+08:00
4,55.5,57.34,74.125,86.00,109.500,433.59,437.5,136.354167,127.948400,24,24:00:00,24,24:00:00,100.0,100.0,2016-02-03T17:00:00Z,2016-02-04T01:00:00+08:00,2016-02-04T16:00:00Z,2016-02-05T00:00:00+08:00


In [ ]:
# Check how complete the expanded metadata columns are.

expanded_missing = air_clean_df[expanded_cols].isna().sum().sort_values(ascending=False)
print("Missing values in expanded metadata columns:")
print(expanded_missing)

Missing values in expanded metadata columns:
summary_sd                     2875
summary_q02                       0
summary_min                       0
summary_q25                       0
summary_median                    0
summary_q98                       0
summary_q75                       0
summary_max                       0
summary_avg                       0
coverage_expectedCount            0
coverage_expectedInterval         0
coverage_observedCount            0
coverage_observedInterval         0
coverage_percentComplete          0
coverage_percentCoverage          0
coverage_datetimeFrom.utc         0
coverage_datetimeFrom.local       0
coverage_datetimeTo.utc           0
coverage_datetimeTo.local         0
dtype: int64


## Create derived air quality datasets

Now that the main air quality dataset has been cleaned, I want to create a few smaller derived datasets that will be easier to use in the final project notebook. These derived datasets will help me analyze pollution at different levels, such as city-wide daily trends, city-wide monthly trends, and differences across stations.

In [ ]:
# Create a city-level daily average dataset by pollutant.
# This summarizes all station readings for each pollutant on each date.

air_daily_city_df = (
    air_clean_df
    .groupby(["date", "year", "month", "season", "parameter"], as_index=False)
    .agg(
        avg_value=("value", "mean"),
        median_value=("value", "median"),
        min_value=("value", "min"),
        max_value=("value", "max"),
        number_of_station_readings=("value", "count"),
        avg_percent_coverage=("coverage_percentCoverage", "mean")
    )
)

print("Shape of daily city-level dataset:", air_daily_city_df.shape)
display(air_daily_city_df.head(10))

Shape of daily city-level dataset: (7146, 11)


,date,year,month,season,parameter,avg_value,median_value,min_value,max_value,number_of_station_readings,avg_percent_coverage
0,2016-01-29,2016,1,Winter,co,1711.142857,1580.00,524.00,3880.0,7,67.000000
1,2016-01-29,2016,1,Winter,no2,63.766667,67.55,35.00,83.9,6,67.000000
2,2016-01-29,2016,1,Winter,o3,15.648571,15.60,1.44,28.2,7,62.142857
3,2016-01-29,2016,1,Winter,pm10,177.662500,136.50,96.30,364.0,8,62.750000
4,2016-01-29,2016,1,Winter,pm25,180.360000,135.00,88.80,298.0,5,60.200000
5,2016-01-29,2016,1,Winter,so2,105.728571,114.00,28.80,198.0,7,62.142857
6,2016-01-30,2016,1,Winter,co,2801.571429,2610.00,851.00,6040.0,7,73.714286
7,2016-01-30,2016,1,Winter,no2,73.383333,81.60,44.30,89.6,6,77.000000
8,2016-01-30,2016,1,Winter,o3,9.886667,7.60,1.42,25.7,6,77.000000
9,2016-01-30,2016,1,Winter,pm10,296.428571,187.00,142.00,876.0,7,67.142857


In [ ]:
# Save the city-level daily summary dataset.

air_daily_city_df.to_csv("data/derived/air_daily_city_pollutant.csv")

In [ ]:
# Create a city-level monthly average dataset by pollutant.
# This will be useful for smoother trend analysis and seasonal comparisons.

air_monthly_city_df = (
    air_clean_df
    .groupby(["year", "month", "season", "parameter"], as_index=False)
    .agg(
        avg_value=("value", "mean"),
        median_value=("value", "median"),
        min_value=("value", "min"),
        max_value=("value", "max"),
        number_of_readings=("value", "count"),
        avg_percent_coverage=("coverage_percentCoverage", "mean")
    )
)

print("Shape of monthly city-level dataset:", air_monthly_city_df.shape)
display(air_monthly_city_df.head(10))

Shape of monthly city-level dataset: (254, 10)


,year,month,season,parameter,avg_value,median_value,min_value,max_value,number_of_readings,avg_percent_coverage
0,2016,1,Winter,co,2341.619048,1710.000,524.00,6130.0,21,72.238095
1,2016,1,Winter,no2,74.711111,80.400,35.00,120.0,18,74.333333
2,2016,1,Winter,o3,12.082500,9.715,1.42,28.2,20,71.700000
3,2016,1,Winter,pm10,237.360870,180.000,96.30,876.0,23,66.304348
4,2016,1,Winter,pm25,242.485714,209.500,88.80,516.0,14,70.571429
5,2016,1,Winter,so2,134.165000,128.500,28.80,286.0,20,67.300000
6,2016,2,Winter,co,2111.455896,1440.000,2.77,16300.0,212,90.344340
7,2016,2,Winter,no2,64.844505,63.050,15.60,128.0,182,90.785714
8,2016,2,Winter,o3,26.687198,18.100,0.00,248.0,197,91.974619
9,2016,2,Winter,pm10,168.936162,141.000,0.00,722.0,271,91.498155


In [ ]:
# Save the city-level monthly summary dataset.

air_monthly_city_df.to_csv("data/derived/air_monthly_city_pollutant.csv", index=False)

In [ ]:
# Create a station-level summary dataset by pollutant.
# This helps compare which stations tend to have higher pollution levels.

air_station_summary_df = (
    air_clean_df
    .groupby(["location_id", "station_name", "parameter"], as_index=False)
    .agg(
        avg_value=("value", "mean"),
        median_value=("value", "median"),
        min_value=("value", "min"),
        max_value=("value", "max"),
        number_of_readings=("value", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        avg_percent_coverage=("coverage_percentCoverage", "mean")
    )
)

print("Shape of station-level summary dataset:", air_station_summary_df.shape)
display(air_station_summary_df.head(10))

Shape of station-level summary dataset: (74, 11)


,location_id,station_name,parameter,avg_value,median_value,min_value,max_value,number_of_readings,first_date,last_date,avg_percent_coverage
0,19,MNB,o3,45.082246,40.60,3.000,248.0,690,2016-01-29,2018-10-18,87.769565
1,19,MNB,pm10,139.516356,117.00,7.830,728.0,999,2016-01-29,2019-03-13,85.314314
2,19,MNB,pm25,96.797518,55.10,5.330,713.0,999,2016-01-29,2019-03-13,85.314314
3,19,MNB,so2,67.516753,40.25,0.000,314.0,770,2016-01-29,2019-03-13,87.903896
4,23,Amgalan,co,1026.613219,631.00,127.000,7310.0,817,2016-01-29,2018-07-25,83.331701
5,23,Amgalan,no2,32.572324,27.70,1.110,90.8,981,2016-01-29,2019-03-13,84.891947
6,23,Amgalan,o3,50.844290,50.50,4.410,110.0,909,2016-01-29,2018-10-31,84.184818
7,23,Amgalan,pm10,123.609246,114.00,3.000,510.0,968,2016-02-02,2019-03-13,84.421488
8,23,Amgalan,pm25,61.971684,39.60,3.000,364.0,968,2016-02-02,2019-03-13,84.429752
9,23,Amgalan,so2,35.817617,16.50,0.333,326.0,951,2016-01-29,2019-03-13,85.310200


In [ ]:
# Save the station-level summary dataset.

air_station_summary_df.to_csv("data/derived/air_station_summary_pollutant.csv", index=False)

In [ ]:
# Check how many cleaned rows we have for each pollutant.

pollutant_counts_df = (
    air_clean_df["parameter"]
    .value_counts()
    .rename_axis("parameter")
    .reset_index(name="number_of_rows")
)

display(pollutant_counts_df)

,parameter,number_of_rows
0,so2,10423
1,pm10,10135
2,no2,8930
3,pm25,7875
4,co,6672
5,o3,6450


## Download and clean weather data

At this point, I have already collected and cleaned the main air quality dataset for Ulaanbaatar. The next step is to prepare the weather dataset so I can later study whether weather conditions are associated with pollution levels.

In [ ]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 47.9077,
	"longitude": 106.8832,
	"start_date": "2016-01-01",
	"end_date": "2019-12-31",
	"daily": ["temperature_2m_max", "temperature_2m_min", "apparent_temperature_mean", "temperature_2m_mean", "precipitation_sum", "wind_speed_10m_mean", "snowfall_sum"],
	"timezone": "Asia/Singapore",
	"timeformat": "unixtime",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_apparent_temperature_mean = daily.Variables(2).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(3).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(4).ValuesAsNumpy()
daily_wind_speed_10m_mean = daily.Variables(5).ValuesAsNumpy()
daily_snowfall_sum = daily.Variables(6).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["apparent_temperature_mean"] = daily_apparent_temperature_mean
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["wind_speed_10m_mean"] = daily_wind_speed_10m_mean
daily_data["snowfall_sum"] = daily_snowfall_sum

weather_df = pd.DataFrame(data = daily_data)
print("\nDaily data\n", weather_df)


Coordinates: 47.90861129760742°N 106.86566925048828°E
Elevation: 1285.0 m asl
Timezone: b'Asia/Singapore'b'GMT+8'
Timezone difference to GMT+0: 28800s

Daily data
                           date  temperature_2m_max  temperature_2m_min  \
0    2016-01-01 00:00:00+08:00           -7.639500          -20.839500   
1    2016-01-02 00:00:00+08:00          -13.939501          -21.039501   
2    2016-01-03 00:00:00+08:00          -16.989500          -23.989500   
3    2016-01-04 00:00:00+08:00          -15.889500          -29.339500   
4    2016-01-05 00:00:00+08:00          -13.339500          -22.039501   
...                        ...                 ...                 ...   
1456 2019-12-27 00:00:00+08:00           -9.421499          -19.321499   
1457 2019-12-28 00:00:00+08:00          -15.971500          -22.321499   
1458 2019-12-29 00:00:00+08:00          -22.371500          -28.621500   
1459 2019-12-30 00:00:00+08:00          -16.721500          -28.721500   
1460 2019-12-31 00:00:

In [ ]:
# Save the initial weather data
weather_df.to_csv("weather_initial.csv", index=False)

In [ ]:
# Basic inspection of the dataset
print("Shape of weather dataset:", weather_df.shape)
print("\nColumns:")
print(weather_df.columns.tolist())

print("\nFirst 5 rows:")
display(weather_df.head())

print("\nData types:")
print(weather_df.dtypes)

Shape of weather dataset: (1461, 8)

Columns:
['date', 'temperature_2m_max', 'temperature_2m_min', 'apparent_temperature_mean', 'temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_mean', 'snowfall_sum']

First 5 rows:


,date,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,temperature_2m_mean,precipitation_sum,wind_speed_10m_mean,snowfall_sum
0,2016-01-01 00:00:00+08:00,-7.639500,-20.839500,-18.472845,-14.191585,1.9,4.341495,1.40
1,2016-01-02 00:00:00+08:00,-13.939501,-21.039501,-21.538406,-16.779085,0.0,6.680271,0.28
2,2016-01-03 00:00:00+08:00,-16.989500,-23.989500,-25.874399,-20.595751,0.0,9.325680,0.14
3,2016-01-04 00:00:00+08:00,-15.889500,-29.339500,-28.470091,-23.118666,0.0,9.306138,0.00
4,2016-01-05 00:00:00+08:00,-13.339500,-22.039501,-23.719477,-18.743670,0.0,7.326762,0.00



Data types:
date                         datetime64[ns, Asia/Singapore]
temperature_2m_max                                  float32
temperature_2m_min                                  float32
apparent_temperature_mean                           float32
temperature_2m_mean                                 float32
precipitation_sum                                   float32
wind_speed_10m_mean                                 float32
snowfall_sum                                        float32
dtype: object


In [ ]:
# Check missing values and create a clean daily date column for the weather data.

missing_summary = weather_df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
print(missing_summary)

weather_clean_df = weather_df.copy()

# Convert the timezone-aware datetime column to a simple date.
weather_clean_df["date"] = pd.to_datetime(weather_clean_df["date"], errors="coerce").dt.date

# Create useful time columns for later analysis.
weather_clean_df["year"] = pd.to_datetime(weather_clean_df["date"]).dt.year
weather_clean_df["month"] = pd.to_datetime(weather_clean_df["date"]).dt.month
weather_clean_df["day"] = pd.to_datetime(weather_clean_df["date"]).dt.day

def month_to_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    elif month in [9, 10, 11]:
        return "Fall"
    return None

weather_clean_df["season"] = weather_clean_df["month"].apply(month_to_season)

print("\nShape after basic weather cleaning:", weather_clean_df.shape)
display(weather_clean_df.head())

Missing values by column:
date                         0
temperature_2m_max           0
temperature_2m_min           0
apparent_temperature_mean    0
temperature_2m_mean          0
precipitation_sum            0
wind_speed_10m_mean          0
snowfall_sum                 0
dtype: int64

Shape after basic weather cleaning: (1461, 12)


,date,temperature_2m_max,temperature_2m_min,apparent_temperature_mean,temperature_2m_mean,precipitation_sum,wind_speed_10m_mean,snowfall_sum,year,month,day,season
0,2016-01-01,-7.639500,-20.839500,-18.472845,-14.191585,1.9,4.341495,1.40,2016,1,1,Winter
1,2016-01-02,-13.939501,-21.039501,-21.538406,-16.779085,0.0,6.680271,0.28,2016,1,2,Winter
2,2016-01-03,-16.989500,-23.989500,-25.874399,-20.595751,0.0,9.325680,0.14,2016,1,3,Winter
3,2016-01-04,-15.889500,-29.339500,-28.470091,-23.118666,0.0,9.306138,0.00,2016,1,4,Winter
4,2016-01-05,-13.339500,-22.039501,-23.719477,-18.743670,0.0,7.326762,0.00,2016,1,5,Winter


## Merge the weather data with the daily city-level air quality data

Now that both datasets are cleaned, the next step is to merge the weather data with the city-level daily air quality dataset. This creates one combined dataset that can be used to study possible relationships between pollution and weather conditions.

In [ ]:
# Check the date ranges of the weather data and the daily city-level air data before merging.

print("Weather date range:")
print("Start:", weather_clean_df["date"].min())
print("End:", weather_clean_df["date"].max())

print("\nAir daily city-level date range:")
print("Start:", air_daily_city_df["date"].min())
print("End:", air_daily_city_df["date"].max())

print("\nWeather rows:", len(weather_clean_df))
print("Air daily city-level rows:", len(air_daily_city_df))

Weather date range:
Start: 2016-01-01
End: 2019-12-31

Air daily city-level date range:
Start: 2016-01-29
End: 2019-12-30

Weather rows: 1461
Air daily city-level rows: 7146


In [ ]:
# Merge the daily city-level air data with the cleaned weather data using the date column.

air_weather_daily_df = air_daily_city_df.merge(
    weather_clean_df,
    on="date",
    how="left",
    suffixes=("", "_weather")
)

print("Shape of merged air + weather dataset:", air_weather_daily_df.shape)
display(air_weather_daily_df.head())

Shape of merged air + weather dataset: (7146, 22)


,date,year,month,season,parameter,avg_value,median_value,min_value,max_value,number_of_station_readings,...,temperature_2m_min,apparent_temperature_mean,temperature_2m_mean,precipitation_sum,wind_speed_10m_mean,snowfall_sum,year_weather,month_weather,day,season_weather
0,2016-01-29,2016,1,Winter,co,1711.142857,1580.00,524.00,3880.0,7,...,-32.239502,-31.814425,-26.518669,0.0,8.496136,0.0,2016,1,29,Winter
1,2016-01-29,2016,1,Winter,no2,63.766667,67.55,35.00,83.9,6,...,-32.239502,-31.814425,-26.518669,0.0,8.496136,0.0,2016,1,29,Winter
2,2016-01-29,2016,1,Winter,o3,15.648571,15.60,1.44,28.2,7,...,-32.239502,-31.814425,-26.518669,0.0,8.496136,0.0,2016,1,29,Winter
3,2016-01-29,2016,1,Winter,pm10,177.662500,136.50,96.30,364.0,8,...,-32.239502,-31.814425,-26.518669,0.0,8.496136,0.0,2016,1,29,Winter
4,2016-01-29,2016,1,Winter,pm25,180.360000,135.00,88.80,298.0,5,...,-32.239502,-31.814425,-26.518669,0.0,8.496136,0.0,2016,1,29,Winter


In [ ]:
# Check missing values in the merged dataset, especially for the weather columns.

merged_missing_summary = air_weather_daily_df.isna().sum().sort_values(ascending=False)
print("Missing values in merged dataset:")
print(merged_missing_summary)

weather_columns = [
    "temperature_2m_max",
    "temperature_2m_min",
    "apparent_temperature_mean",
    "temperature_2m_mean",
    "precipitation_sum",
    "wind_speed_10m_mean",
    "snowfall_sum"
]


Missing values in merged dataset:
date                          0
year                          0
month                         0
season                        0
parameter                     0
avg_value                     0
median_value                  0
min_value                     0
max_value                     0
number_of_station_readings    0
avg_percent_coverage          0
temperature_2m_max            0
temperature_2m_min            0
apparent_temperature_mean     0
temperature_2m_mean           0
precipitation_sum             0
wind_speed_10m_mean           0
snowfall_sum                  0
year_weather                  0
month_weather                 0
day                           0
season_weather                0
dtype: int64


In [ ]:
# Save the final merged air and weather dataset for use in the main project notebook.

air_weather_daily_df.to_csv("data/derived/air_weather_daily_merged.csv", index=False)
weather_clean_df.to_csv("data/clean/ulaanbaatar_weather_clean.csv", index=False)

In [ ]:
# Re-save the final expanded clean air dataset so the saved file matches the latest version in memory.
air_clean_df.to_csv("data/clean/ulaanbaatar_air_daily_clean.csv", index=False)